# Fase 5 — Baseline

Questo notebook documenta e riproduce integralmente la Fase 5 del progetto **AI_microtradingAUX**.

Obiettivi:
- costruire baseline semplici: **always flat, always long, random, momentum, EMA crossover**;
- usare per tutte le strategie lo **stesso execution engine** della Fase 4;
- calcolare le metriche di performance;
- confrontare ogni baseline **senza costi** e **con costi**;
- produrre `reports/baselines.csv` e `reports/figures/baseline_equity.png`;
- verificare che non venga introdotto look-ahead nelle strategie che usano il prezzo.

## 1. Setup e dipendenze

Il notebook è pensato per essere eseguito dalla root del repository. Il modulo `baselines.py` contiene la logica definitiva e viene importato qui per mantenere notebook e pipeline allineati.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'baselines.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
      
from baselines import (
    STRATEGIES,
    load_config,
    load_test_data,
    build_target_positions,
    make_execution_config,
    calculate_metrics,
    run_baselines,
    save_equity_figure,
)

config = load_config()
data = load_test_data(config)
print(f'Dataset: {len(data):,} barre')
print(f'Periodo: {data.index.min()} -> {data.index.max()}')
print(f'Strategie: {STRATEGIES}')

## 2. Dataset utilizzato

La Fase 5 utilizza il dataset processed disponibile, con priorità a `xauusd_m1_test.parquet`; in assenza di questo vengono considerati i fallback già definiti in `baselines.py`.

La colonna `Close` è il prezzo mid utilizzato dall'execution engine.

In [ ]:
display(data.head())
display(data[['Close']].describe())
assert 'Close' in data.columns
assert not data.empty
assert data['Close'].notna().all()

## 3. Definizione delle baseline

**Always flat:** posizione target sempre `0`.

**Always long:** posizione target sempre `+1`.

**Random:** posizione casuale tra `-1`, `0`, `+1`, con seed del progetto per rendere il risultato riproducibile.

**Momentum:** `sign(Close[t-1] - Close[t-2])`; la posizione è ritardata di una barra per evitare di usare il `Close` della barra di esecuzione.

**EMA crossover:** `sign(EMA12 - EMA26)` ritardato di una barra; anche qui il segnale usato dall'ordine non contiene informazione futura.

In [ ]:
positions = build_target_positions(data, seed=int(config['project']['seed']))
positions_df = pd.DataFrame(positions)
display(positions_df.head(10))

for name, series in positions.items():
    assert set(series.dropna().unique()).issubset({-1, 0, 1})
    print(f'{name:15s} | flat={(series == 0).mean():.2%} | long={(series == 1).mean():.2%} | short={(series == -1).mean():.2%}')

## 4. Stesso execution engine per tutte le strategie

Il requisito architetturale della Fase 5 è che le baseline **non abbiano un backtester separato**. Ogni strategia passa attraverso `simulate_portfolio_over_series`, che a sua volta utilizza la catena di esecuzione/portfolio della Fase 4.

Per il confronto dei costi vengono usate due configurazioni: tutti i costi a zero e i valori presenti in `config.yaml`.

In [ ]:
from gold_rl.simulation import simulate_portfolio_over_series
from gold_rl.execution import ExecutionCostsConfig

environment = config['environment']
no_costs = make_execution_config(environment, with_costs=False)
with_costs = make_execution_config(environment, with_costs=True)

print('Configurazione senza costi:', no_costs)
print('Configurazione con costi:   ', with_costs)

# Smoke test: anche questa simulazione passa dallo stesso engine usato da tutte le baseline.
test_target = positions['always_flat'].to_numpy(dtype=int)
simulation = simulate_portfolio_over_series(
    mid_prices=data['Close'],
    target_positions=test_target,
    config=no_costs,
    initial_balance=float(environment['initial_balance']),
)
display(simulation.head())

## 5. Metriche

Per ogni coppia `(strategia, costi)` vengono calcolati:
- equity iniziale e finale;
- total return;
- maximum drawdown;
- volatilità per barra;
- Sharpe annualizzato per dati M1;
- numero di trade;
- turnover.

Il calcolo è effettuato sull'equity prodotta dall'execution engine, quindi include gli effetti dei costi quando la modalità è `with_costs`.

In [ ]:
report = run_baselines(data, config)
report = report.sort_values(['strategy', 'costs']).reset_index(drop=True)
display(report)

expected_columns = {
    'initial_equity', 'final_equity', 'total_return', 'max_drawdown',
    'volatility_per_bar', 'sharpe_annualized_m1', 'trades', 'turnover',
    'strategy', 'costs', 'bars', 'start', 'end', 'seed'
}
assert expected_columns.issubset(report.columns)
assert len(report) == len(STRATEGIES) * 2
assert set(report['costs']) == {'without_costs', 'with_costs'}

## 6. Confronto senza costi vs con costi

La tabella seguente rende esplicito l'effetto dell'introduzione di spread, commissioni, slippage e turnover penalty configurati nell'ambiente.

In [ ]:
comparison = report.pivot(index='strategy', columns='costs', values=['final_equity', 'total_return', 'max_drawdown', 'trades', 'turnover'])
display(comparison)

return_comparison = report.pivot(index='strategy', columns='costs', values='total_return')
return_comparison['cost_impact'] = return_comparison['with_costs'] - return_comparison['without_costs']
display(return_comparison.sort_values('cost_impact'))

## 7. Output della Fase 5

Gli output ufficiali sono:
- `reports/baselines.csv`: risultati numerici per tutte le baseline e le due modalità di costo;
- `reports/figures/baseline_equity.png`: equity curve, con pannello senza costi e pannello con costi.

In [ ]:
reports_dir = PROJECT_ROOT / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)
csv_path = reports_dir / 'baselines.csv'
report.to_csv(csv_path, index=False, float_format='%.12g')
save_equity_figure(data, config)

figure_path = reports_dir / 'figures' / 'baseline_equity.png'
print('CSV:', csv_path)
print('Figura:', figure_path)
assert csv_path.exists()
assert figure_path.exists()

In [ ]:
# Anteprima della figura generata dalla pipeline.
from IPython.display import Image, display
display(Image(filename=str(figure_path)))

## 8. Verifiche finali

Checklist di completamento della Fase 5:
- [x] 5 strategie baseline;
- [x] seed riproducibile per la random baseline;
- [x] segnali momentum/EMA ritardati di una barra;
- [x] stesso execution engine per tutte le strategie;
- [x] confronto con e senza costi;
- [x] metriche di performance;
- [x] report CSV;
- [x] figura delle equity curve;
- [x] test automatici dedicati alla Fase 5.

In [ ]:
print('Fase 5 completata.')
print(f'Strategie: {len(STRATEGIES)}')
print(f'Righe report: {len(report)}')
print(f'Output CSV presente: {csv_path.exists()}')
print(f'Output PNG presente: {figure_path.exists()}')